projection onto `h` and `r` from the english rounds of the harmful and harmless prompts translated to other languages

- we picked `h` at layer 12 on instruction's last token and `r` at layer 18 on last template token to be effective, so we use that direction

In [ ]:
import json
from pathlib import Path

import torch
from harm_route.models import load_bridge, load_tokenizer, model_config, pad_token_id

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   
name = "qwen3-1.7b"
kw = model_config(name)["template_kwargs"]
tok = load_tokenizer(name)
bridge, pad_id = load_bridge(name, device="cpu"), pad_token_id(tok)          # float32 on CPU, as in notebook 02
dirs = torch.load(ROOT / f"data/cache/e00/{name}_directions.pt")             # {"h": [L, d], "r": [P, L, d]}
h_hat = dirs["h"][12] / dirs["h"][12].norm()                                 # [d], unit length
r_hat = dirs["r"][-1, 18] / dirs["r"][-1, 18].norm()                         # [d]
print(dirs["h"].shape, dirs["r"].shape, h_hat.norm().item(), r_hat.norm().item())   # norms = 1.0

loop over languages

In [ ]:
from harm_route.collect import collect_resid_batched
from harm_route.prompts import find_positions

langs = ["eng_Latn", "npi_Deva", "srp_Cyrl", "srp_Latn"]
sets = ["harmful_val", "harmless_val"]
H = dirs["h"] / dirs["h"].norm(dim=-1, keepdim=True)                          # [L, d] / [L, 1] -> [L, d], unit per layer
R = dirs["r"][-1] / dirs["r"][-1].norm(dim=-1, keepdim=True)                  # [L, d], last post-inst position
s_h, s_r = {}, {}      # s_h[lang][set] -> [N]      fixed layer (h L12, r L18)
ph, pr = {}, {}        # ph[lang][set]  -> [N, L]   every layer, on that layer's own English direction
acts = {}              # acts[lang][set] -> (inst [N, L, d], last [N, L, d]), val activations
for lang in langs:
    data = json.load(open(ROOT / f"data/raw/e01/{name}/splits/{lang}.json"))
    s_h[lang], s_r[lang], ph[lang], pr[lang], acts[lang] = {}, {}, {}, {}, {}
    for st in sets:
        ps = [find_positions(tok, p, kw) for p in data[st]]
        inst, post = collect_resid_batched(bridge, ps, pad_id)               # [N, L, d], [N, P, L, d]
        inst, last = inst.float(), post[:, -1].float()                       # [N, L, d], [N, L, d]
        s_h[lang][st] = inst[:, 12] @ h_hat                                  # [N, d] @ [d] -> [N]
        s_r[lang][st] = last[:, 18] @ r_hat                                  # [N]
        ph[lang][st] = torch.einsum("nld,ld->nl", inst, H)                   # [N, L]
        pr[lang][st] = torch.einsum("nld,ld->nl", last, R)                   # [N, L]
        acts[lang][st] = (inst, last)

def scale(x, s):
    """English scale: 0 = mean English harmless, 1 = mean English harmful."""
    lo, hi = s["eng_Latn"]["harmless_val"].mean(), s["eng_Latn"]["harmful_val"].mean()
    return (x - lo) / (hi - lo)

table: mean projection per language on the English scale (eng harmful = 1, eng harmless = 0)

In [ ]:
print(f"{'lang':9s} {'h harmful':>9s} {'h harmless':>10s} {'r harmful':>9s} {'r harmless':>10s}")
for lang in langs:
    print(f"{lang:9s} " + " ".join(f"{scale(s[lang][st], s).mean():{w}.2f}"
          for s in (s_h, s_r) for st, w in zip(sets, (9, 10))))

per request: English (x) vs other language (y), harmful prompts. On the diagonal = same as English; below = weaker

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, s, title in zip(axes, (s_h, s_r), ("h (L12, t_inst)", "r (L18, last post-inst)")):
    x = scale(s["eng_Latn"]["harmful_val"], s)
    for lang in langs[1:]:
        ax.scatter(x, scale(s[lang]["harmful_val"], s), s=15, label=lang)
    ax.axline((0, 0), slope=1, color="k", lw=0.8)                            # y = x: same as English
    ax.axhline(0, color="gray", lw=0.5, ls="--")                             # English harmless level
    ax.set_title(title); ax.set_xlabel("English (scaled)"); ax.set_ylabel("other language (scaled)")
axes[0].legend(); plt.tight_layout(); plt.show()

Layer sweep, part 1: separation along the English direction at every layer

Residual norms grow with depth, so raw gaps aren't comparable across layers. Use a scale-free separation:
d' = (mean harmful - mean harmless) / pooled std, per layer (each layer's own English `h` / `r`).
d' = 0: harmful and harmless overlap completely; d' > 2: clearly separated.

In [ ]:
def sep(p, lang):
    a, b = p[lang]["harmful_val"], p[lang]["harmless_val"]                    # [N, L] each
    pooled = ((a.var(0) + b.var(0)) / 2).sqrt()                               # [L]
    return (a.mean(0) - b.mean(0)) / pooled                                   # [L]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (p, nm) in zip(axes, [(ph, "h (t_inst)"), (pr, "r (last post-inst)")]):
    for lang in langs:
        ax.plot(sep(p, lang), label=lang)
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.set_title(f"separation along English {nm}"); ax.set_xlabel("layer")
axes[0].set_ylabel("d\' (harmful vs harmless)"); axes[0].legend(); plt.tight_layout(); plt.show()

Own-language directions from TRAIN, tested on VAL (no circularity)

Train prompts: Google MT, not reviewed, chrF >= 40 (data/raw/e01/{model}/train/splits). Same recipe as English E0:
own h[l] = mean(harmful_train) - mean(harmless_train) at t_inst, own r[l] the same at the last post-inst token.

In [ ]:
train_mean = {}                                                              # [lang][set] -> (inst [L, d], last [L, d])
for lang in langs:
    data = json.load(open(ROOT / f"data/raw/e01/{name}/train/splits/{lang}.json"))
    train_mean[lang] = {}
    for st in ["harmful_train", "harmless_train"]:
        ps = [find_positions(tok, p, kw) for p in data[st]]
        inst, post = collect_resid_batched(bridge, ps, pad_id)               # [N, L, d], [N, P, L, d]
        train_mean[lang][st] = (inst.float().mean(0), post[:, -1].float().mean(0))   # [L, d] each

own = {lang: {k: train_mean[lang]["harmful_train"][j] - train_mean[lang]["harmless_train"][j]   # [L, d]
              for j, k in enumerate(["h", "r"])} for lang in langs}
cos = torch.nn.functional.cosine_similarity
# sanity: English own h/r = the stored E0 directions (same train prompts, minus 2 harmless) -> ~1.0 at every layer
print("eng own h vs E0 h, min cos:", cos(own["eng_Latn"]["h"], dirs["h"], dim=-1).min().item())
print("eng own r vs E0 r, min cos:", cos(own["eng_Latn"]["r"], dirs["r"][-1], dim=-1).min().item())

Left: d' on VAL along the language's OWN direction (solid) vs the English direction (dashed).
Own >> English means the model separates harm in this language, but along a different direction.
Right: cosine of own direction vs English (both from ~130 train prompts).

In [ ]:
def sep_along(lang, k, D):
    """d' of val harmful vs harmless along unit directions D [L, d], for k = 'h' (t_inst) or 'r' (last post-inst)"""
    j = 0 if k == "h" else 1
    D = D / D.norm(dim=-1, keepdim=True)                                       # [L, d]
    a = torch.einsum("nld,ld->nl", acts[lang]["harmful_val"][j], D)           # [N, L]
    b = torch.einsum("nld,ld->nl", acts[lang]["harmless_val"][j], D)
    return (a.mean(0) - b.mean(0)) / ((a.var(0) + b.var(0)) / 2).sqrt()     # [L]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for row, k in enumerate(["h", "r"]):
    for i, lang in enumerate(langs):
        col = f"C{i}"
        axes[row, 0].plot(sep_along(lang, k, own[lang][k]), color=col, label=f"{lang} own")
        axes[row, 0].plot(sep_along(lang, k, own["eng_Latn"][k]), color=col, ls="--")
        axes[row, 1].plot(cos(own[lang][k], own["eng_Latn"][k], dim=-1), color=col, label=lang)
    axes[row, 0].set_ylabel(f"{k}: d' on val (solid own, dashed English)")
    axes[row, 1].set_ylabel(f"{k}: cos(own, English)")
for ax in axes[1]:
    ax.set_xlabel("layer")
axes[0, 0].legend(); plt.tight_layout(); plt.show()